In [6]:
import sys
import os

# 1. Chuyển vị trí làm việc ra thư mục gốc của dự án
# Lệnh này kiểm tra nếu đang ở trong folder 'notebooks' thì nhảy ra ngoài 1 cấp
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

# 2. Thêm thư mục gốc vào đường dẫn hệ thống để import module
project_root = os.getcwd()
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Vị trí làm việc hiện tại: {os.getcwd()}")

Vị trí làm việc hiện tại: d:\HCMUT\NMTTNT\ai-ambulance-coordinator


In [7]:
import os
import sys
from dotenv import load_dotenv

# 1. Quản lý đường dẫn: Đảm bảo Notebook luôn đứng ở thư mục gốc dự án
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# 2. Nạp biến môi trường từ file .env (Nếu chạy ở Local)
load_dotenv()

# 3. Cấu hình Kaggle API Credentials
# os.getenv sẽ tìm trong .env trước. Nếu không thấy (như trên Colab), 
# nó sẽ lấy từ biến môi trường hệ thống.
os.environ['KAGGLE_USERNAME'] = os.getenv('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = os.getenv('KAGGLE_KEY')

# Kiểm tra nhanh (không in ra Key để bảo mật)
if os.environ.get('KAGGLE_KEY'):
    print(f"✅ Cấu hình thành công! Đang chạy tại: {os.getcwd()}")
    print(f"👤 User: {os.environ.get('KAGGLE_USERNAME')}")
else:
    print("❌ Lỗi: Không tìm thấy Kaggle Credentials. Hãy kiểm tra file .env hoặc Colab Secrets.")

✅ Cấu hình thành công! Đang chạy tại: d:\HCMUT\NMTTNT\ai-ambulance-coordinator
👤 User: hoanganh1105


In [8]:
from modules.data_loader import fetch_and_load_data

# Fetch the dataset from Kaggle directly to the Colab instance
df = fetch_and_load_data()

if df is not None:
    print(f"Dataset successfully loaded. Shape: {df.shape}")
    display(df.head())

Successfully located and loaded data from: data\Final_Augmented_dataset_Diseases_and_Symptoms.csv
Dataset successfully loaded. Shape: (246945, 378)


,diseases,anxiety and nervousness,depression,shortness of breath,depressive or psychotic symptoms,sharp chest pain,dizziness,insomnia,abnormal involuntary movements,chest tightness,...,stuttering or stammering,problems with orgasm,nose deformity,lump over jaw,sore in nose,hip weakness,back swelling,ankle stiffness or tightness,ankle weakness,neck weakness
0,panic disorder,1,0,1,1,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
1,panic disorder,0,0,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
2,panic disorder,1,1,1,1,0,1,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,panic disorder,1,0,0,1,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,panic disorder,1,1,0,0,0,0,1,1,1,...,0,0,0,0,0,0,0,0,0,0


In [9]:
from modules.preprocessor import preprocess_and_save_features

# Process raw data into binary symptom matrices and target labels
# The results will be saved into the 'features/' directory
X, y, symptoms_vocab, le = preprocess_and_save_features(df, output_dir="features")

print("\nPreprocessing Results:")
print(f"- Symptoms detected: {len(symptoms_vocab)}")
print(f"- Disease classes: {len(le.classes_)}")
print(f"- Feature matrix saved to: features/X_symptoms.npy")


Preprocessing Results:
- Symptoms detected: 377
- Disease classes: 773
- Feature matrix saved to: features/X_symptoms.npy


In [10]:
from modules.ml_model import DiseaseClassifier
import numpy as np

# Initialize and train the Machine Learning model
classifier = DiseaseClassifier(model_dir="features")
classifier.train(X_path="features/X_symptoms.npy", y_path="features/y_diseases.npy")

# --- Simulate an Emergency Call (AI Diagnostics Test) ---
print("\n--- Testing Phase 1: Emergency Call Simulation ---")
dummy_patient = np.zeros(len(symptoms_vocab), dtype=np.int8)

# Simulating symptoms: 'anxiety and nervousness' and 'depression'
dummy_patient[0] = 1 
dummy_patient[1] = 1

predicted_disease = classifier.predict_disease(dummy_patient)
print(f"Input Symptoms: {symptoms_vocab[0]}, {symptoms_vocab[1]}")
print(f"AI Prediction Outcome: {predicted_disease}")

Loading feature matrices for training...
Data successfully loaded. X shape: (246945, 377), y shape: (246945,)
Training Decision Tree Classifier... (This might take a minute without depth limits)

[TRAINING COMPLETE] Model Accuracy on test set: 81.44%
Model successfully saved to features\decision_tree_model.joblib

--- Testing Phase 1: Emergency Call Simulation ---
Input Symptoms: anxiety and nervousness, depression
AI Prediction Outcome: drug withdrawal
